# Sales Performance Analysis: Where the Revenue Comes From

**Business question:** which products, categories, regions, and customers actually drive revenue - and where should the business focus next?

**Approach:** I treat this like a consulting engagement. First I audit the data, then I follow the money through four lenses - growth, mix, customers, and operations - and I close with prioritized recommendations. Every claim below is computed from the data in this notebook.

**Dataset:** 9,800 retail order lines (`train.csv`), January 2015 to December 2018, covering category, region, customer segment, ship mode, and geography.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NAVY, TEAL, TEAL_DK, SLATE, LIGHT, AMBER = (
    "#1b2a4a", "#14b8a6", "#0f766e", "#64748b", "#e2e8f0", "#d97706")
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": LIGHT, "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": SLATE, "ytick.color": SLATE, "font.size": 10,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlecolor": NAVY,
})

df = pd.read_csv("../data/train.csv")
df["order_date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y")
df["ship_date"] = pd.to_datetime(df["Ship Date"], format="%d/%m/%Y")
df["fulfill_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
print(f"Loaded {len(df):,} rows, {df['Order ID'].nunique():,} orders, "
      f"${df['Sales'].sum():,.0f} total revenue.")

## 1. Data quality audit

Before any analysis, I check what I am working with: missing values, duplicates, date sanity, and the grain of the data.

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDate range:", df["order_date"].min().date(), "->", df["order_date"].max().date())
print("Negative sales rows:", (df["Sales"] <= 0).sum())
print("\nGrain check - rows per order:")
print(df.groupby("Order ID").size().describe()[["mean", "50%", "max"]].round(2))

**Audit result:** the data is clean. 11 missing postal codes (irrelevant for revenue analysis), zero duplicates, dates parse cleanly across 2015-2018, no negative sales. Average order holds about 2 line items. I can proceed.

## 2. Growth: revenue up 50%, driven by order volume

Revenue grew from $480k (2015) to $722k (2018). Orders grew faster than revenue (947 to 1,661, up 75%), which means growth came from selling to more orders, not from bigger baskets. That distinction matters for strategy.

In [ ]:
ys = df.groupby("year")["Sales"].sum()
yo = df.groupby("year")["Order ID"].nunique()
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.bar(ys.index.astype(str), ys.values / 1000, color=TEAL, width=0.55)
for x, v in zip(ys.index.astype(str), ys.values):
    ax1.text(x, v / 1000 + 8, f"${v/1000:.0f}k", ha="center",
             fontsize=9, color=NAVY, weight="bold")
ax1.set_ylabel("Revenue ($ thousands)")
ax1.set_title("Revenue grew 50% in four years, driven by order volume (+75%)")
ax2 = ax1.twinx()
ax2.plot(yo.index.astype(str), yo.values, color=NAVY, marker="o", linewidth=2.5)
ax2.set_ylabel("Orders", color=NAVY)
ax1.set_xlabel("Year")
plt.tight_layout(); plt.show()
print(f"Revenue CAGR 2015-2018: {(ys.loc[2018]/ys.loc[2015])**(1/3)-1:.1%}")
print(f"Average order value 2015: ${ys.loc[2015]/yo.loc[2015]:.0f} -> "
      f"2018: ${ys.loc[2018]/yo.loc[2018]:.0f}")

**Takeaway:** a healthy, volume-led growth story. Average order value actually slipped slightly ($507 to $435), so the business is winning on reach, not on ticket size.

## 3. Category mix: balanced revenue, but very different engines

Technology leads revenue at 36.6%, but the mix is surprisingly balanced. The real story is underneath: Office Supplies generates 60% of all transactions but only 31% of revenue (small tickets, high volume), while Technology does the reverse. And Office Supplies is growing fastest at 17.1% CAGR - it is the hidden growth engine.

In [ ]:
cs = df.groupby("Category")["Sales"].sum()
fig, ax = plt.subplots(figsize=(7, 4.5))
wedges, _, autotexts = ax.pie(
    cs.values,
    labels=[f"{i}\n${v/1000:.0f}k" for i, v in zip(cs.index, cs.values)],
    colors=[NAVY, TEAL, AMBER], autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2),
    pctdistance=0.78, textprops=dict(fontsize=10))
for t in autotexts:
    t.set_color("white"); t.set_weight("bold"); t.set_fontsize(9)
ax.set_title("Revenue is balanced across categories - Technology leads at 36.6%")
plt.tight_layout(); plt.show()

yr_cat = df.groupby(["year", "Category"])["Sales"].sum().unstack()
cagr = ((yr_cat.loc[2018] / yr_cat.loc[2015]) ** (1/3) - 1).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(cagr.index, cagr.values * 100,
               color=[TEAL_DK, TEAL, NAVY], height=0.5)
for b, v in zip(bars, cagr.values):
    ax.text(v * 100 + 0.3, b.get_y() + b.get_height() / 2, f"{v:.1%}",
            va="center", fontsize=10, weight="bold", color=NAVY)
ax.set_xlim(0, cagr.max() * 100 + 4)
ax.set_xlabel("Compound annual growth rate, 2015-2018")
ax.set_title("Office Supplies is the fastest grower at 17.1% CAGR")
plt.tight_layout(); plt.show()
print(f"Office Supplies: {100*(df['Category']=='Office Supplies').mean():.1f}% "
      f"of transactions, "
      f"{100*df.loc[df['Category']=='Office Supplies','Sales'].sum()/df['Sales'].sum():.1f}% "
      f"of revenue")

## 4. Seasonality: the second half carries the year

November runs at 186% of an average month; February at just 32%. Q4 alone delivers roughly double a normal quarter, consistently across all four years. Inventory, staffing, and marketing budgets should be shaped around this curve, not spread evenly.

In [ ]:
ms = df.groupby("month")["Sales"].sum() / df["year"].nunique()
idx = ms / ms.mean() * 100
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, ax = plt.subplots(figsize=(10, 4.5))
colors = [AMBER if v >= 150 else (TEAL if v >= 100 else SLATE)
          for v in idx.values]
bars = ax.bar(months, idx.values, color=colors, width=0.65, edgecolor="white")
ax.axhline(100, color=NAVY, linestyle="--", linewidth=1)
for b, v in zip(bars, idx.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 4, f"{v:.0f}",
            ha="center", fontsize=9, color=NAVY)
ax.set_ylabel("Index (100 = average month)")
ax.set_title("November runs at 186% of an average month - H2 carries the year")
plt.tight_layout(); plt.show()
q = df.groupby(df["order_date"].dt.to_period("Q"))["Sales"].sum()
print("Q4 share of annual revenue by year:")
for y in [2015, 2016, 2017, 2018]:
    yr = q[[str(p) for p in q.index if str(p).startswith(str(y))]]
    print(f"  {y}: {yr.loc[f'{y}Q4']/yr.sum():.1%}")

## 5. Customers: fiercely loyal, barely growing the base

780 of 793 customers (98%) ordered more than once, and repeat buyers drive 99.8% of revenue. Revenue concentration is healthy (top 10 customers = 6.8%). The flip side: almost no new-customer revenue. Retention is a solved problem here; acquisition is the gap.

In [ ]:
cust = df.groupby("Customer ID").agg(orders=("Order ID", "nunique"),
                                       sales=("Sales", "sum"))
repeat = cust[cust["orders"] > 1]
print(f"Customers: {len(cust)}, repeat buyers: {len(repeat)} "
      f"({len(repeat)/len(cust):.1%})")
print(f"Revenue from repeat buyers: "
      f"{repeat['sales'].sum()/cust['sales'].sum():.1%}")
cs_sorted = cust["sales"].sort_values(ascending=False)
for p in [10, 50]:
    print(f"Top {p} customers: {cs_sorted.head(p).sum()/cs_sorted.sum():.1%} "
          f"of revenue")
seg = df.groupby("Segment").agg(orders=("Order ID", "nunique"),
                                sales=("Sales", "sum"))
seg["aov"] = (seg["sales"] / seg["orders"]).round(0)
print("\nRevenue and average order value by segment:")
print(seg)

**Takeaway:** average order value is nearly identical across Consumer ($453), Corporate ($462), and Home Office ($475) - no segment commands a premium, so growth must come from volume or mix, not from segment targeting on price.

## 6. Geography: the real concentration risk

Product revenue is well diversified (top 10 products = only 10.8% of sales), but geography is not: California and New York together drive a third of all revenue. The West leads on volume while the East leads on order value ($489 average ticket).

In [ ]:
st = df.groupby("State")["Sales"].sum().sort_values().tail(10)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(st.index, st.values / 1000, color=NAVY, height=0.6,
               edgecolor="white")
for b, v in zip(bars, st.values):
    ax.text(v / 1000 + 6, b.get_y() + b.get_height() / 2, f"${v/1000:.0f}k",
            va="center", fontsize=9, color=NAVY)
ax.set_xlabel("Revenue ($ thousands)")
ax.set_title("California + New York = one third of all revenue")
plt.tight_layout(); plt.show()

ro = df.groupby("Region").agg(orders=("Order ID", "nunique"),
                              sales=("Sales", "sum"))
ro["aov"] = (ro["sales"] / ro["orders"]).round(0)
ro = ro.sort_values("sales")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2))
a1.barh(ro.index, ro["sales"] / 1000, color=TEAL, height=0.55)
a1.set_xlabel("Revenue ($ thousands)")
a1.set_title("Revenue by region")
a2.barh(ro.index, ro["aov"], color=NAVY, height=0.55)
for i, v in enumerate(ro["aov"]):
    a2.text(v + 6, i, f"${v:.0f}", va="center", fontsize=9, color=NAVY)
a2.set_xlabel("Average order value ($)")
a2.set_title("East has the highest ticket at $489")
fig.suptitle("West leads on volume, East leads on order value",
             fontsize=13, weight="bold", color=NAVY)
plt.tight_layout(); plt.show()
top2 = df.groupby("State")["Sales"].sum().sort_values(ascending=False).head(2).sum()
print(f"CA + NY share of total revenue: {top2/df['Sales'].sum():.1%}")

## 7. Operations: fulfilment promises are kept

Delivery times form a clean ladder - Same Day averages 0.04 days, First Class 2.2, Second Class 3.2, Standard 5.0 - and no mode ever breaches its promise window. 59% of revenue ships Standard, so customers mostly trade speed for cost. Operations is not the problem; it is a quiet strength.

In [ ]:
fm = df.groupby("Ship Mode")["fulfill_days"].agg(["mean", "median"])
order = ["Same Day", "First Class", "Second Class", "Standard Class"]
fm = fm.loc[order]
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar(fm.index, fm["mean"],
              color=[TEAL, TEAL_DK, NAVY, SLATE], width=0.55,
              edgecolor="white")
for b, v in zip(bars, fm["mean"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.08, f"{v:.1f} days",
            ha="center", fontsize=10, weight="bold", color=NAVY)
ax.set_ylabel("Average days, order to shipment")
ax.set_title("Fulfilment ladder is clean - every mode beats its promise")
ax.set_ylim(0, fm["mean"].max() + 1)
plt.tight_layout(); plt.show()
mix = df.groupby("Ship Mode")["Sales"].sum() / df["Sales"].sum() * 100
print("Revenue share by ship mode:")
for m in order:
    print(f"  {m}: {mix[m]:.1f}%")

## 8. Sub-category leaders

Phones ($328k) and Chairs ($323k) top the sub-category ranking - one from Technology, one from Furniture. The top 10 sub-categories span all three categories, which confirms the diversification story: there is no single-category dependency.

In [ ]:
sc = df.groupby("Sub-Category")["Sales"].sum().sort_values().tail(10)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(sc.index, sc.values / 1000, color=TEAL, height=0.6,
               edgecolor="white")
for b, v in zip(bars, sc.values):
    ax.text(v / 1000 + 4, b.get_y() + b.get_height() / 2, f"${v/1000:.0f}k",
            va="center", fontsize=9, color=NAVY)
ax.set_xlabel("Revenue ($ thousands)")
ax.set_title("Phones and Chairs are the top sub-categories")
plt.tight_layout(); plt.show()

## 9. Recommendations

Ranked by expected impact:

1. **Fund the H2 surge, don't spread spend evenly.** November runs at 186% of an average month. Shift inventory, staffing, and marketing budget toward September-December instead of a flat annual plan.

2. **Invest behind Office Supplies.** It is 60% of transactions, growing at 17.1% CAGR (fastest of all categories), yet only 31% of revenue. Small lifts in attachment rate or average ticket here compound fast.

3. **Fix acquisition.** 99.8% of revenue comes from repeat buyers - retention is solved, but the customer base is not growing. Even a modest new-customer program diversifies future revenue.

4. **De-risk California and New York.** One third of revenue sits in two states. Priority: replicate what works there (assortment, service levels) in Texas and Washington, the next tier.

5. **Protect the East's ticket premium.** East has the highest average order value ($489 vs $426 in Central). Understand what drives it - mix, segment, or service - before pushing volume plays that could dilute it.

## 10. Limitations and next steps

- The dataset has **no profit, discount, or quantity fields**, so margin analysis is impossible here; with cost data, the next step would be a discount-vs-margin study (high discounts often destroy profit in retail).
- **No returns data**, so I cannot measure return rates by category or product.
- Customer analysis is limited to order counts; with timestamps per customer, a full RFM segmentation would sharpen the acquisition recommendation.
- All figures are nominal USD, 2015-2018; conclusions describe this retailer, not retail in general.